# Part II-B: Recurrent Neural Network (LSTM) for Surface Crack Classification

## Overview
This notebook implements a Long Short-Term Memory (LSTM) network for classifying cracked vs non-cracked surfaces. We treat each image as a sequence by processing it row-by-row (or using column-wise patches).

## Approach
- Images are resized and treated as sequences of rows
- Each row (with all 3 color channels) is one time step in the sequence
- LSTM processes the sequence and the final hidden state is used for classification

## Hyperparameters to Explore
- **Batch size**: 16, 32, 64
- **Number of LSTM layers**: 1, 2, 3
- **Hidden size**: 64, 128, 256
- **Dropout rate**: 0.0, 0.3, 0.5
- **Optimizer**: SGD, Adam, RMSProp
- **L2 Regularization**: 0.0, 1e-4, 1e-3
- **Learning rate**: 0.1, 0.01, 0.001
- **Learning rate scheduler**: StepLR, ReduceLROnPlateau

### Import Libraries
Import PyTorch modules including the LSTM layer, along with data utilities and visualization libraries.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import pandas as pd
import time
import copy

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

### Configure Device
Check for GPU availability to accelerate LSTM training.

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.backends.cudnn.benchmark = True
print("CUDNN Benchmark Enabled (Eager Mode Optimized)")
torch.backends.cudnn.benchmark = True
print("PyTorch Eager Mode: Active (CUDNN benchmark enabled)")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


### Define Custom Dataset Class
Same dataset class as before - loads images from Cracked and Non Cracked folders.

In [3]:
class SurfaceCrackDataset(Dataset):
    def __init__(self, cracked_dir, non_cracked_dir, transform=None, max_samples=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        cracked_files = sorted(os.listdir(cracked_dir))
        if max_samples:
            cracked_files = cracked_files[:max_samples]
        for fname in cracked_files:
            self.image_paths.append(os.path.join(cracked_dir, fname))
            self.labels.append(1)
        
        non_cracked_files = sorted(os.listdir(non_cracked_dir))
        if max_samples:
            non_cracked_files = non_cracked_files[:max_samples]
        for fname in non_cracked_files:
            self.image_paths.append(os.path.join(non_cracked_dir, fname))
            self.labels.append(0)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

### Set Up Data Transforms and Loaders
Resize images to a smaller size (64x64) to make LSTM training computationally feasible. The LSTM will process 64 time steps (rows), each with 64*3=192 features.

In [4]:
CRACKED_DIR = 'Cracked'
NON_CRACKED_DIR = 'Non Cracked'
IMAGE_SIZE = 64

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = SurfaceCrackDataset(CRACKED_DIR, NON_CRACKED_DIR, transform=train_transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size],
                                            generator=torch.Generator().manual_seed(42))
test_dataset.dataset.transform = test_transform

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")
print(f"Sequence length (rows): {IMAGE_SIZE}")
print(f"Input features per step (cols x channels): {IMAGE_SIZE * 3}")

Training samples: 76873
Testing samples: 19219
Sequence length (rows): 64
Input features per step (cols x channels): 192


### Define LSTM Model
Build an LSTM-based model that processes images row-by-row as sequences. The model stacks multiple LSTM layers and uses the final hidden state for classification through fully connected layers.

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes=2, dropout_rate=0.0, bidirectional=False):
        super(LSTMClassifier, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        
        fc_input = hidden_size * self.num_directions
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(fc_input, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        seq_len = x.size(2)
        x = x.permute(0, 2, 3, 1).contiguous().view(batch_size, seq_len, -1)
        
        h0 = torch.zeros(self.num_layers * self.num_directions, batch_size, self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * self.num_directions, batch_size, self.hidden_size).to(x.device)
        
        lstm_out, _ = self.lstm(x, (h0, c0))
        
        last_hidden = lstm_out[:, -1, :]
        
        output = self.classifier(last_hidden)
        return output

input_size = IMAGE_SIZE * 3
seq_len = IMAGE_SIZE
print(f"LSTM input size per time step: {input_size}")
print(f"Sequence length: {seq_len}")

LSTM input size per time step: 192
Sequence length: 64


### Define Training Function
Training function for one epoch - computes forward pass, loss, backpropagation, and weight updates.

In [6]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, correct / total

### Define Evaluation Function
Evaluates the model on a given dataset, returning loss, overall accuracy, and per-class accuracy.

In [7]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    class_correct = [0, 0]
    class_total = [0, 0]
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            for i in range(labels.size(0)):
                label = labels[i].item()
                class_total[label] += 1
                class_correct[label] += predicted[i].eq(label).item()
    
    class_accuracies = [c/t if t > 0 else 0 for c, t in zip(class_correct, class_total)]
    return running_loss / total, correct / total, class_accuracies

### Define Full Training Loop
Complete training loop with learning rate scheduling, best model tracking, and history recording.

In [8]:
def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler,
                num_epochs, device, model_name='lstm'):
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'test_acc_cracked': [], 'test_acc_non_cracked': []
    }
    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc, test_class_acc = evaluate(model, test_loader, criterion, device)
        
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(test_loss)
            else:
                scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['test_acc_cracked'].append(test_class_acc[1])
        history['test_acc_non_cracked'].append(test_class_acc[0])
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch+1}/{num_epochs}] LR: {current_lr:.6f} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')
    
    model.load_state_dict(best_model_wts)
    return model, history, best_acc

### Helper: Plot Training History
Visualize loss and accuracy curves across epochs.

In [9]:
def plot_history(history, title='Training History'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['test_loss'], label='Test Loss', marker='s')
    axes[0].set_title('Loss Over Epochs', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
    axes[1].plot(history['test_acc'], label='Test Accuracy', marker='s')
    axes[1].set_title('Accuracy Over Epochs', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

### Experiment 1: Baseline LSTM Configuration
Start with a baseline: 2 LSTM layers, 128 hidden units, dropout=0.3, Adam optimizer, lr=0.001.

In [10]:
NUM_EPOCHS = 15

model = LSTMClassifier(
    input_size=input_size, hidden_size=128, num_layers=2,
    dropout_rate=0.3, bidirectional=False
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Starting baseline LSTM training...")

baseline_model, baseline_history, baseline_acc = train_model(
    model, train_loader, test_loader, criterion, optimizer, None,
    NUM_EPOCHS, device, 'lstm_baseline'
)

print(f"\nBest test accuracy: {baseline_acc:.4f}")
plot_history(baseline_history, 'LSTM Baseline (2 layers, hidden=128, Adam, lr=0.001)')

Model parameters: 313,986
Starting baseline LSTM training...


RuntimeError: permute(sparse_coo): number of dimensions in the tensor input does not match the length of the desired ordering of dimensions i.e. input.dim() = 4 is not equal to len(dims) = 3

### Experiment 2: Effect of Number of LSTM Layers
Compare 1, 2, and 3 LSTM layers. More layers can capture more complex temporal patterns but may overfit.

In [ ]:
layer_configs = {
    '1 layer': 1,
    '2 layers': 2,
    '3 layers': 3
}

layer_results = {}

for name, num_layers in layer_configs.items():
    print(f"\n{'='*50}")
    print(f"Training with {name}")
    print(f"{'='*50}")
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=128, num_layers=num_layers,
        dropout_rate=0.3, bidirectional=False
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'lstm_{name}'
    )
    layer_results[name] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy for {name}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in layer_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of LSTM Layers on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 3: Effect of Hidden Size
Compare hidden sizes of 64, 128, and 256. Larger hidden sizes give the LSTM more capacity but increase computation.

In [ ]:
hidden_configs = {
    'hidden=64': 64,
    'hidden=128': 128,
    'hidden=256': 256
}

hidden_results = {}

for name, hidden_size in hidden_configs.items():
    print(f"\n{'='*50}")
    print(f"Training with {name}")
    print(f"{'='*50}")
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=hidden_size, num_layers=2,
        dropout_rate=0.3, bidirectional=False
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'lstm_{name}'
    )
    hidden_results[name] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy for {name}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in hidden_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Hidden Size on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 4: Effect of Dropout Rate
Test dropout rates of 0.0, 0.3, and 0.5. Dropout helps prevent overfitting in the LSTM and classifier layers.

In [ ]:
dropout_rates = [0.0, 0.3, 0.5]
dropout_results = {}

for rate in dropout_rates:
    print(f"\n{'='*50}")
    print(f"Training with dropout={rate}")
    print(f"{'='*50}")
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=128, num_layers=2,
        dropout_rate=rate, bidirectional=False
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'lstm_dropout_{rate}'
    )
    dropout_results[f'dropout={rate}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with dropout={rate}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in dropout_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Dropout Rate on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 5: Effect of Optimizer
Compare SGD, Adam, and RMSProp for LSTM training.

In [ ]:
optimizers_config = {
    'SGD': lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    'Adam': lambda params: optim.Adam(params, lr=0.001),
    'RMSProp': lambda params: optim.RMSprop(params, lr=0.001)
}

optimizer_results = {}

for name, opt_fn in optimizers_config.items():
    print(f"\n{'='*50}")
    print(f"Training with {name} optimizer")
    print(f"{'='*50}")
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=128, num_layers=2,
        dropout_rate=0.3, bidirectional=False
    ).to(device)
    optimizer = opt_fn(model.parameters())
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'lstm_{name}'
    )
    optimizer_results[name] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with {name}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in optimizer_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Optimizer on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 6: Effect of L2 Regularization
Test weight decay values of 0.0, 1e-4, and 1e-3.

In [ ]:
weight_decays = [0.0, 1e-4, 1e-3]
wd_results = {}

for wd in weight_decays:
    print(f"\n{'='*50}")
    print(f"Training with weight_decay={wd}")
    print(f"{'='*50}")
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=128, num_layers=2,
        dropout_rate=0.3, bidirectional=False
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=wd)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'lstm_wd_{wd}'
    )
    wd_results[f'wd={wd}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with wd={wd}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in wd_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of L2 Regularization on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 7: Effect of Learning Rate
Compare learning rates of 0.1, 0.01, and 0.001.

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
lr_results = {}

for lr in learning_rates:
    print(f"\n{'='*50}")
    print(f"Training with lr={lr}")
    print(f"{'='*50}")
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=128, num_layers=2,
        dropout_rate=0.3, bidirectional=False
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'lstm_lr_{lr}'
    )
    lr_results[f'lr={lr}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with lr={lr}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in lr_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Learning Rate on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 8: Learning Rate Schedulers
Compare no scheduler, StepLR, and ReduceLROnPlateau.

In [ ]:
scheduler_configs = {
    'No Scheduler': None,
    'StepLR': lambda opt: optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5),
    'ReduceLROnPlateau': lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
}

scheduler_results = {}

for name, sched_fn in scheduler_configs.items():
    print(f"\n{'='*50}")
    print(f"Training with {name}")
    print(f"{'='*50}")
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=128, num_layers=2,
        dropout_rate=0.3, bidirectional=False
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = sched_fn(optimizer) if sched_fn else None
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, scheduler,
        NUM_EPOCHS, device, f'lstm_sched_{name}'
    )
    scheduler_results[name] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with {name}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in scheduler_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Learning Rate Scheduler on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 9: Effect of Batch Size
Compare batch sizes of 16, 32, and 64.

In [ ]:
batch_sizes = [16, 32, 64]
bs_results = {}

for bs in batch_sizes:
    print(f"\n{'='*50}")
    print(f"Training with batch_size={bs}")
    print(f"{'='*50}")
    
    train_loader_exp = DataLoader(train_dataset, batch_size=bs, shuffle=True, num_workers=0)
    test_loader_exp = DataLoader(test_dataset, batch_size=bs, shuffle=False, num_workers=0)
    
    model = LSTMClassifier(
        input_size=input_size, hidden_size=128, num_layers=2,
        dropout_rate=0.3, bidirectional=False
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    
    _, history, best_acc = train_model(
        model, train_loader_exp, test_loader_exp, criterion, optimizer, scheduler,
        NUM_EPOCHS, device, f'lstm_bs_{bs}'
    )
    bs_results[f'bs={bs}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with batch_size={bs}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in bs_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Batch Size on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Final Model: Best Configuration
Train the final LSTM model using the best hyperparameters discovered.

In [ ]:
BEST_CONFIG = {
    'hidden_size': 128,
    'num_layers': 2,
    'dropout_rate': 0.3,
    'bidirectional': False,
    'optimizer': 'Adam',
    'learning_rate': 0.001,
    'weight_decay': 1e-4,
    'batch_size': 32,
    'scheduler': 'StepLR',
    'num_epochs': 20
}

print("Training final LSTM model with best configuration:")
for k, v in BEST_CONFIG.items():
    print(f"  {k}: {v}")

train_loader_final = DataLoader(train_dataset, batch_size=BEST_CONFIG['batch_size'], shuffle=True, num_workers=0)
test_loader_final = DataLoader(test_dataset, batch_size=BEST_CONFIG['batch_size'], shuffle=False, num_workers=0)

final_model = LSTMClassifier(
    input_size=input_size,
    hidden_size=BEST_CONFIG['hidden_size'],
    num_layers=BEST_CONFIG['num_layers'],
    dropout_rate=BEST_CONFIG['dropout_rate'],
    bidirectional=BEST_CONFIG['bidirectional']
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_model.parameters(), lr=BEST_CONFIG['learning_rate'],
                       weight_decay=BEST_CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

final_model, final_history, final_acc = train_model(
    final_model, train_loader_final, test_loader_final, criterion, optimizer, scheduler,
    BEST_CONFIG['num_epochs'], device, 'lstm_final'
)

torch.save(final_model.state_dict(), 'lstm_best_model.pth')
print(f"\nFinal model saved. Best test accuracy: {final_acc:.4f}")
plot_history(final_history, 'LSTM Final Model (Best Configuration)')

### Comprehensive Results Summary
Compile all experimental results into a summary table.

In [ ]:
results_summary = []

results_summary.append({'Experiment': 'Baseline', 'Config': '2 layers, hidden=128, Adam, lr=0.001', 'Best Acc': f"{baseline_acc:.4f}"})

for name, result in layer_results.items():
    results_summary.append({'Experiment': 'LSTM Layers', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in hidden_results.items():
    results_summary.append({'Experiment': 'Hidden Size', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in dropout_results.items():
    results_summary.append({'Experiment': 'Dropout', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in optimizer_results.items():
    results_summary.append({'Experiment': 'Optimizer', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in wd_results.items():
    results_summary.append({'Experiment': 'Weight Decay', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in lr_results.items():
    results_summary.append({'Experiment': 'Learning Rate', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in scheduler_results.items():
    results_summary.append({'Experiment': 'Scheduler', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in bs_results.items():
    results_summary.append({'Experiment': 'Batch Size', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

results_summary.append({'Experiment': 'Final', 'Config': 'Best combined config', 'Best Acc': f"{final_acc:.4f}"})

df_results = pd.DataFrame(results_summary)
print(df_results.to_string(index=False))

### Final Evaluation on Train and Test Sets
Report final model performance on both training and test sets.

In [ ]:
print("="*60)
print("FINAL LSTM MODEL EVALUATION")
print("="*60)

train_loss, train_acc, train_class_acc = evaluate(final_model, train_loader_final, criterion, device)
test_loss, test_acc, test_class_acc = evaluate(final_model, test_loader_final, criterion, device)

print(f"\nTraining Set:")
print(f"  Loss: {train_loss:.4f}")
print(f"  Accuracy: {train_acc:.4f}")
print(f"  Cracked Accuracy: {train_class_acc[1]:.4f}")
print(f"  Non-Cracked Accuracy: {train_class_acc[0]:.4f}")

print(f"\nTest Set:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  Cracked Accuracy: {test_class_acc[1]:.4f}")
print(f"  Non-Cracked Accuracy: {test_class_acc[0]:.4f}")

gap = train_acc - test_acc
print(f"\nTrain-Test Gap: {gap:.4f} {'(Overfitting)' if gap > 0.05 else '(Good generalization)'}")